In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession\
                    .builder\
                    .master("spark://spark-master:7077")\
                    .appName("Day_6_Assignments")\
                    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/18 11:29:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [25]:
#question on lead ,lag()
product_data = [
                    (1,"iphone","01-01-2023",1500000),
                    (2,"samsung","01-01-2023",1100000),
                    (3,"oneplus","01-01-2023",1100000),
                    (1,"iphone","01-02-2023",1300000),
                    (2,"samsung","01-02-2023",1120000),
                    (3,"oneplus","01-02-2023",1120000),
                    (1,"iphone","01-03-2023",1600000),
                    (2,"samsung","01-03-2023",1080000),
                    (3,"oneplus","01-03-2023",1160000),
                    (1,"iphone","01-04-2023",1700000),
                    (2,"samsung","01-04-2023",1800000),
                    (3,"oneplus","01-04-2023",1170000),
                    (1,"iphone","01-05-2023",1200000),
                    (2,"samsung","01-05-2023",980000),
                    (3,"oneplus","01-05-2023",1175000),
                    (1,"iphone","01-06-2023",1100000),
                    (2,"samsung","01-06-2023",1100000),
                    (3,"oneplus","01-06-2023",1200000)
                ]
product_schema = StructType(
                            [
                                StructField('product_id',IntegerType(),True),
                                StructField('product_name',StringType(),True),
                                StructField('sale_date',StringType(),True),
                                StructField('sales',IntegerType(),True)
                            ]
                             
                            )



df = spark.createDataFrame(product_data,schema=product_schema)

#convert date_type

df = df.withColumn('sale_date',to_date('sale_date','dd-MM-yyyy'))

df.show(5)

#1.what is the percentage of loss or gain in sales in each product based on previous month sales

window_spec = Window.partitionBy('product_name').orderBy(col('sale_date').asc())

previous_month_df = df.withColumn('previous_month_sales',lag(col('sales'),1).over(window_spec))
previous_month_df.show(5)

previous_month_df.withColumn('%loss_or_gain',round(((col('sales')-col('previous_month_sales'))/col('sales'))*100,2)).show()

+----------+------------+----------+-------+
|product_id|product_name| sale_date|  sales|
+----------+------------+----------+-------+
|         1|      iphone|2023-01-01|1500000|
|         2|     samsung|2023-01-01|1100000|
|         3|     oneplus|2023-01-01|1100000|
|         1|      iphone|2023-02-01|1300000|
|         2|     samsung|2023-02-01|1120000|
+----------+------------+----------+-------+
only showing top 5 rows

+----------+------------+----------+-------+--------------------+
|product_id|product_name| sale_date|  sales|previous_month_sales|
+----------+------------+----------+-------+--------------------+
|         1|      iphone|2023-01-01|1500000|                NULL|
|         1|      iphone|2023-02-01|1300000|             1500000|
|         1|      iphone|2023-03-01|1600000|             1300000|
|         1|      iphone|2023-04-01|1700000|             1600000|
|         1|      iphone|2023-05-01|1200000|             1700000|
+----------+------------+----------+------

In [44]:
#2 what is the %age of sale of each month based on last 6months sales

window_spec = Window.partitionBy('product_name').orderBy(col('sale_date').asc())
# if data is not there(NULL),then 0
last_6_months_sales = df.withColumn('last_1m_sales',lag('sales',1,0).over(window_spec))\
                        .withColumn('last_2m_sales',lag('sales',2,0).over(window_spec))\
                        .withColumn('last_3m_sales',lag('sales',3,0).over(window_spec))\
                        .withColumn('last_4m_sales',lag('sales',4,0).over(window_spec))\
                        .withColumn('last_5m_sales',lag('sales',5,0).over(window_spec))\
                        .withColumn('total_6m_sales',col('sales')+col('last_1m_sales')+col('last_2m_sales')+col('last_3m_sales')+col('last_4m_sales')+col('last_5m_sales'))

last_6_months_sales.show(7,truncate=False)

df_final = last_6_months_sales.withColumn('%sales_based_on_last_6m',round((col('sales')/col('total_6m_sales'))*100,2) )\
                                .drop(col('last_1m_sales'),col('last_2m_sales'),col('last_3m_sales'),col('last_4m_sales'),col('last_5m_sales'))\
                                .show()

#-----------using row between----------------

window_spec = Window.partitionBy('product_id')\
                    .orderBy('sale_date')\
                    .rowsBetween(-5,0)

df_result = df.withColumn('total_6m_sales',sum('sales').over(window_spec))\
                .withColumn('pct_sales',round((col('sales')/col('total_6m_sales'))*100,2))\
                .show()

+----------+------------+----------+-------+-------------+-------------+-------------+-------------+-------------+--------------+
|product_id|product_name|sale_date |sales  |last_1m_sales|last_2m_sales|last_3m_sales|last_4m_sales|last_5m_sales|total_6m_sales|
+----------+------------+----------+-------+-------------+-------------+-------------+-------------+-------------+--------------+
|1         |iphone      |2023-01-01|1500000|0            |0            |0            |0            |0            |1500000       |
|1         |iphone      |2023-02-01|1300000|1500000      |0            |0            |0            |0            |2800000       |
|1         |iphone      |2023-03-01|1600000|1300000      |1500000      |0            |0            |0            |4400000       |
|1         |iphone      |2023-04-01|1700000|1600000      |1300000      |1500000      |0            |0            |6100000       |
|1         |iphone      |2023-05-01|1200000|1700000      |1600000      |1300000      |1500

26/04/18 12:47:31 ERROR TaskSchedulerImpl: Lost executor 1 on 172.18.0.5: worker lost: Not receiving heartbeat for 60 seconds
26/04/18 12:47:31 ERROR TaskSchedulerImpl: Lost executor 0 on 172.18.0.4: worker lost: Not receiving heartbeat for 60 seconds


In [7]:
orders_schema_struct = StructType(
                                    [
                                        StructField("order_id",IntegerType()),
                                        StructField("customer_id",IntegerType()),
                                        StructField("product_id",IntegerType()),
                                        StructField("price",FloatType()),
                                        StructField("order_date",DateType()),
                                        StructField("order_status",StringType()),
                                        StructField("state",StringType()),
                                        StructField("quantity",IntegerType()),
                                    ]
                                )

df_orders = spark.read.csv('/data/orders_50mb.csv',schema=orders_schema_struct,header=True)

df_orders.show(5)

df_orders.createOrReplaceTempView('orders_table')

NameError: name 'strucType' is not defined

In [5]:
#product table
product_schema_struct = StructType(
                                    [
                                    StructField('product_id',IntegerType()),
                                    StructField('product_name',StringType()),
                                    StructField('cost_price',FloatType())
                                    ]
                                )

df_product = spark.read.csv('/data/products.csv',schema=product_schema_struct,header=True)

df_product.show(5)

df_product.createOrReplaceTempView('product_table')

[Stage 1:>                                                          (0 + 1) / 1]

+----------+--------------------+----------+
|product_id|        product_name|cost_price|
+----------+--------------------+----------+
|         1|      Wireless Mouse|     120.0|
|         2| Mechanical Keyboard|     135.0|
|         3|       USB C Charger|     150.0|
|         4|        Laptop Stand|     165.0|
|         5|Noise Cancelling ...|     180.0|
+----------+--------------------+----------+
only showing top 5 rows



In [12]:
df= df_orders.filter(col('order_status')=='DELIVERED')\
            .join(df_product,'product_id','inner')\
            .withColumn('profit',(col('price')-col('cost_price'))*col('quantity'))\
            .groupBy('state','product_name').agg(sum('profit').alias('product_profit'))

window_spec= Window.partitionBy('state').orderBy(col('product_profit').desc())

df.withColumn('rank',rank().over(window_spec))\
            .filter(col('rank')<=2)\
            .show()

+----------------+-------------------+------------------+----+
|           state|       product_name|    product_profit|rank|
+----------------+-------------------+------------------+----+
|       Karnataka|     Wireless Mouse| 6680.639930725098|   1|
|       Karnataka|   Wireless Earbuds| 3602.089988708496|   2|
|          Odisha|Mechanical Keyboard|5389.3100509643555|   1|
|          Odisha|     Wireless Mouse| 4880.380027770996|   2|
|          Kerala|   Wireless Earbuds| 5972.230010986328|   1|
|          Kerala|     Wireless Mouse| 5906.629943847656|   2|
|      Tamil Nadu|     Wireless Mouse| 6527.709930419922|   1|
|      Tamil Nadu|   Wireless Earbuds| 3214.830146789551|   2|
|    Chhattisgarh|     Wireless Mouse| 8475.800010681152|   1|
|    Chhattisgarh|   Wireless Earbuds|4655.8599853515625|   2|
|  Andhra Pradesh|   Wireless Earbuds| 5988.870079040527|   1|
|  Andhra Pradesh|     Wireless Mouse|5575.1599044799805|   2|
|  Madhya Pradesh|     Wireless Mouse| 6755.16993713378

26/01/31 15:15:26 ERROR TaskSchedulerImpl: Lost executor 1 on 172.18.0.4: worker lost: Not receiving heartbeat for 60 seconds
26/01/31 15:15:26 ERROR TaskSchedulerImpl: Lost executor 0 on 172.18.0.6: worker lost: Not receiving heartbeat for 60 seconds
26/01/31 15:46:28 ERROR TaskSchedulerImpl: Lost executor 2 on 172.18.0.6: worker lost: Not receiving heartbeat for 60 seconds
26/01/31 15:46:28 ERROR TaskSchedulerImpl: Lost executor 3 on 172.18.0.4: worker lost: Not receiving heartbeat for 60 seconds
26/01/31 16:10:30 ERROR TaskSchedulerImpl: Lost executor 4 on 172.18.0.6: worker lost: Not receiving heartbeat for 60 seconds
26/01/31 16:10:30 ERROR TaskSchedulerImpl: Lost executor 5 on 172.18.0.4: worker lost: Not receiving heartbeat for 60 seconds
26/01/31 16:51:34 ERROR TaskSchedulerImpl: Lost executor 6 on 172.18.0.4: worker lost: Not receiving heartbeat for 60 seconds
26/01/31 16:51:34 ERROR TaskSchedulerImpl: Lost executor 7 on 172.18.0.6: worker lost: Not receiving heartbeat for 60 

In [12]:
df= df_orders.filter(col('order_status')=='DELIVERED')\
            .join(df_product,'product_id','inner')\
            .withColumn('profit',(col('price')-col('cost_price'))*col('quantity'))\
            .groupBy('state','product_name').agg(sum('profit').alias('product_profit'))

window_spec= Window.partitionBy('state').orderBy(col('product_profit').desc())

df.withColumn('rank',rank().over(window_spec))\
            .filter(col('rank')<=2)\
            .show()

+----------------+-------------------+------------------+----+
|           state|       product_name|    product_profit|rank|
+----------------+-------------------+------------------+----+
|       Karnataka|     Wireless Mouse| 6680.639930725098|   1|
|       Karnataka|   Wireless Earbuds| 3602.089988708496|   2|
|          Odisha|Mechanical Keyboard|5389.3100509643555|   1|
|          Odisha|     Wireless Mouse| 4880.380027770996|   2|
|          Kerala|   Wireless Earbuds| 5972.230010986328|   1|
|          Kerala|     Wireless Mouse| 5906.629943847656|   2|
|      Tamil Nadu|     Wireless Mouse| 6527.709930419922|   1|
|      Tamil Nadu|   Wireless Earbuds| 3214.830146789551|   2|
|    Chhattisgarh|     Wireless Mouse| 8475.800010681152|   1|
|    Chhattisgarh|   Wireless Earbuds|4655.8599853515625|   2|
|  Andhra Pradesh|   Wireless Earbuds| 5988.870079040527|   1|
|  Andhra Pradesh|     Wireless Mouse|5575.1599044799805|   2|
|  Madhya Pradesh|     Wireless Mouse| 6755.16993713378

26/01/31 15:15:26 ERROR TaskSchedulerImpl: Lost executor 1 on 172.18.0.4: worker lost: Not receiving heartbeat for 60 seconds
26/01/31 15:15:26 ERROR TaskSchedulerImpl: Lost executor 0 on 172.18.0.6: worker lost: Not receiving heartbeat for 60 seconds
26/01/31 15:46:28 ERROR TaskSchedulerImpl: Lost executor 2 on 172.18.0.6: worker lost: Not receiving heartbeat for 60 seconds
26/01/31 15:46:28 ERROR TaskSchedulerImpl: Lost executor 3 on 172.18.0.4: worker lost: Not receiving heartbeat for 60 seconds
26/01/31 16:10:30 ERROR TaskSchedulerImpl: Lost executor 4 on 172.18.0.6: worker lost: Not receiving heartbeat for 60 seconds
26/01/31 16:10:30 ERROR TaskSchedulerImpl: Lost executor 5 on 172.18.0.4: worker lost: Not receiving heartbeat for 60 seconds
26/01/31 16:51:34 ERROR TaskSchedulerImpl: Lost executor 6 on 172.18.0.4: worker lost: Not receiving heartbeat for 60 seconds
26/01/31 16:51:34 ERROR TaskSchedulerImpl: Lost executor 7 on 172.18.0.6: worker lost: Not receiving heartbeat for 60 

In [35]:
#spark.sql

spark.sql("""with agg_cte AS
            (
                select o.state,
                        p.product_name,
                        sum((o.price-p.cost_price)*o.quantity) as product_profit
                from orders_table o
                join(product_table p)
                on o.product_id = p.product_id
                where o.order_status='DELIVERED'
                group by state,product_name
            ),
            ranked_cte AS
                (select *,dense_rank() over(partition by state order by product_profit desc) as rank
                from agg_cte
                )
            select * from ranked_cte where rank<=2
            
           


"""
         ).show()



+----------------+-------------------+------------------+----+
|           state|       product_name|    product_profit|rank|
+----------------+-------------------+------------------+----+
|       Karnataka|     Wireless Mouse| 6680.639930725098|   1|
|       Karnataka|   Wireless Earbuds| 3602.089988708496|   2|
|          Odisha|Mechanical Keyboard|5389.3100509643555|   1|
|          Odisha|     Wireless Mouse| 4880.380027770996|   2|
|          Kerala|   Wireless Earbuds| 5972.230010986328|   1|
|          Kerala|     Wireless Mouse| 5906.629943847656|   2|
|      Tamil Nadu|     Wireless Mouse| 6527.709930419922|   1|
|      Tamil Nadu|   Wireless Earbuds| 3214.830146789551|   2|
|    Chhattisgarh|     Wireless Mouse| 8475.800010681152|   1|
|    Chhattisgarh|   Wireless Earbuds|4655.8599853515625|   2|
|  Andhra Pradesh|   Wireless Earbuds| 5988.870079040527|   1|
|  Andhra Pradesh|     Wireless Mouse|5575.1599044799805|   2|
|  Madhya Pradesh|     Wireless Mouse| 6755.16993713378

In [39]:
df_orders.show(10)

+--------+-----------+----------+------+----------+------------+-----------+--------+
|order_id|customer_id|product_id| price|order_date|order_status|      state|quantity|
+--------+-----------+----------+------+----------+------------+-----------+--------+
|       1|     192520|        70|152.33|2007-10-10|      PLACED|     Odisha|       2|
|       2|     835421|        34|117.42|2008-12-02|     SHIPPED|     Kerala|       3|
|       3|     159165|        13|128.85|2010-04-22|   DELIVERED|    Gujarat|       3|
|       4|     403890|        25|195.71|2007-05-30|   CANCELLED|      Bihar|       1|
|       5|     273746|        41|125.08|2003-12-13|     SHIPPED|     Odisha|       3|
|       6|     944219|        41|188.95|2002-05-15|   DELIVERED|  Rajasthan|       2|
|       7|     445844|        28|117.61|2009-08-04|     SHIPPED|      Assam|       3|
|       8|     978829|         3|174.42|2005-12-21|      PLACED|      Assam|       3|
|       9|     929531|        78|185.21|2002-11-25|   

In [46]:
df_test = df_orders.orderBy(col('customer_id'),col('order_id').desc()).show()

+--------+-----------+----------+------+----------+------------+----------------+--------+
|order_id|customer_id|product_id| price|order_date|order_status|           state|quantity|
+--------+-----------+----------+------+----------+------------+----------------+--------+
|  861286|          1|        98|146.71|2017-01-16|      PLACED|           Bihar|       2|
|  671992|          1|        90| 100.7|2013-04-13|    RETURNED|     Uttarakhand|       3|
|  549678|          1|        78| 123.8|2008-01-17|      PLACED|       Telangana|       2|
|  986008|          2|        31|106.89|2014-05-06|      PLACED|    Chhattisgarh|       3|
|  285378|          4|        91| 146.8|2023-10-20|   DELIVERED|          Odisha|       3|
|  331740|          5|        86| 121.9|2007-07-16|     SHIPPED|     West Bengal|       3|
|   84347|          5|        75|104.52|2013-05-08|      PLACED|    Chhattisgarh|       2|
|   78457|          6|        46|101.66|2003-12-14|     SHIPPED|     West Bengal|       1|

In [60]:
window_spec = Window.partitionBy('customer_id')
window_spec2 = Window.partitionBy('customer_id').orderBy(col('order_id').desc())

df = df_orders.withColumn('row_num',row_number().over(window_spec2))\
                .withColumn('avg_order_price',avg(col('price')).over(window_spec))\
                .withColumn('price_diff',col('price')-col('avg_order_price'))\
                .filter(col('row_num')==1)\
                .select('customer_id','order_id',col('price').alias('unit_price'),'avg_order_price','price_diff')\
                .orderBy(col('customer_id').asc())\
                .show()

[Stage 137:===========================================>         (164 + 4) / 200]

+-----------+--------+----------+------------------+-------------------+
|customer_id|order_id|unit_price|   avg_order_price|         price_diff|
+-----------+--------+----------+------------------+-------------------+
|          1|  861286|    146.71|123.73666890462239| 22.973337809244796|
|          2|  986008|    106.89|106.88999938964844|                0.0|
|          4|  285378|     146.8| 146.8000030517578|                0.0|
|          5|  331740|     121.9|113.20999908447266|   8.69000244140625|
|          6|   78457|    101.66|101.66000366210938|                0.0|
|          7|  724152|    192.03|192.02999877929688|                0.0|
|          8|   40314|     138.2| 138.1999969482422|                0.0|
|          9|  798291|    197.93|197.92999267578125|                0.0|
|         10|  641398|    173.53|173.52999877929688|                0.0|
|         11|  663797|    191.46| 191.4600067138672|                0.0|
|         12|  794724|    142.79|170.44499969482422

In [67]:
spark.sql(
            """
            with
                latest_cte AS 
                    (select *,
                        row_number() over(partition by customer_id order by order_id desc) as rn
                        from orders_table),
                agg_cte AS
                     (select customer_id,avg(price) as avg_price from latest_cte 
                         group by customer_id
                         order by customer_id asc
                    )
                select l.customer_id,l.order_id,l.price,a.avg_price ,(l.price-a.avg_price) as price_diff
                from latest_cte l 
                inner join agg_cte a
                on l.customer_id=a.customer_id
                where rn =1
                order by l.customer_id asc

            """
).show()

[Stage 146:===============================================>     (179 + 5) / 200]

+-----------+--------+------+------------------+-------------------+
|customer_id|order_id| price|         avg_price|         price_diff|
+-----------+--------+------+------------------+-------------------+
|          1|  861286|146.71|123.73666890462239| 22.973337809244796|
|          2|  986008|106.89|106.88999938964844|                0.0|
|          4|  285378| 146.8| 146.8000030517578|                0.0|
|          5|  331740| 121.9|113.20999908447266|   8.69000244140625|
|          6|   78457|101.66|101.66000366210938|                0.0|
|          7|  724152|192.03|192.02999877929688|                0.0|
|          8|   40314| 138.2| 138.1999969482422|                0.0|
|          9|  798291|197.93|197.92999267578125|                0.0|
|         10|  641398|173.53|173.52999877929688|                0.0|
|         11|  663797|191.46| 191.4600067138672|                0.0|
|         12|  794724|142.79|170.44499969482422|-27.655006408691406|
|         14|  809021|146.31|126.2

26/03/03 10:02:28 ERROR TaskSchedulerImpl: Lost executor 1 on 172.18.0.4: worker lost: Not receiving heartbeat for 60 seconds
26/03/03 10:02:28 ERROR TaskSchedulerImpl: Lost executor 0 on 172.18.0.6: worker lost: Not receiving heartbeat for 60 seconds
26/03/03 10:19:50 ERROR TaskSchedulerImpl: Lost executor 3 on 172.18.0.4: worker lost: Not receiving heartbeat for 60 seconds
26/03/03 10:19:50 ERROR TaskSchedulerImpl: Lost executor 2 on 172.18.0.6: worker lost: Not receiving heartbeat for 60 seconds
26/03/03 10:40:23 ERROR TaskSchedulerImpl: Lost executor 5 on 172.18.0.4: worker lost: Not receiving heartbeat for 60 seconds
26/03/03 10:40:23 ERROR TaskSchedulerImpl: Lost executor 4 on 172.18.0.6: worker lost: Not receiving heartbeat for 60 seconds
26/03/03 11:13:36 ERROR TaskSchedulerImpl: Lost executor 6 on 172.18.0.6: worker lost: Not receiving heartbeat for 60 seconds
26/03/03 11:13:36 ERROR TaskSchedulerImpl: Lost executor 7 on 172.18.0.4: worker lost: Not receiving heartbeat for 60 

In [69]:
df_orders.show(5)
df_product.show(5)

+--------+-----------+----------+------+----------+------------+-------+--------+
|order_id|customer_id|product_id| price|order_date|order_status|  state|quantity|
+--------+-----------+----------+------+----------+------------+-------+--------+
|       1|     192520|        70|152.33|2007-10-10|      PLACED| Odisha|       2|
|       2|     835421|        34|117.42|2008-12-02|     SHIPPED| Kerala|       3|
|       3|     159165|        13|128.85|2010-04-22|   DELIVERED|Gujarat|       3|
|       4|     403890|        25|195.71|2007-05-30|   CANCELLED|  Bihar|       1|
|       5|     273746|        41|125.08|2003-12-13|     SHIPPED| Odisha|       3|
+--------+-----------+----------+------+----------+------------+-------+--------+
only showing top 5 rows

+----------+--------------------+----------+
|product_id|        product_name|cost_price|
+----------+--------------------+----------+
|         1|      Wireless Mouse|     120.0|
|         2| Mechanical Keyboard|     135.0|
|         3|

In [83]:
df_orders.withColumn('prod_price',col('price')*col('quantity'))\
            .join(df_product,'product_id','inner')\
            .groupBy('product_id','product_name',year(col('order_date')).alias('year'))\
                .agg(sum('prod_price').alias('year_prod_revenue'))\
            .orderBy(col('year').asc(),col('product_id').asc())\
            .show(100)

+----------+--------------------+----+------------------+
|product_id|        product_name|year| year_prod_revenue|
+----------+--------------------+----+------------------+
|         1|      Wireless Mouse|2000| 132608.4299468994|
|         2| Mechanical Keyboard|2000|114651.05010986328|
|         3|       USB C Charger|2000|130809.52007293701|
|         4|        Laptop Stand|2000|116124.37001037598|
|         5|Noise Cancelling ...|2000|121751.84001159668|
|         6|   Bluetooth Speaker|2000|123047.68002319336|
|         7| External Hard Drive|2000|121485.27979278564|
|         8|           Webcam HD|2000| 128041.7499923706|
|         9|    Gaming Mouse Pad|2000| 124675.7700958252|
|        10|   Smartphone Tripod|2000|128465.19031524658|
|        11|    Wireless Earbuds|2000|125684.42973327637|
|        12| Power Bank 10000mAh|2000|116500.15975952148|
|        13|          HDMI Cable|2000|121058.29975891113|
|        14|USB Flash Drive 64GB|2000|126338.91049194336|
|        15|  

In [99]:
df_orders.withColumn('prod_price',col('price')*col('quantity'))\
            .join(df_product,'product_id','inner')\
            .groupBy('product_id','product_name',year(col('order_date')).alias('year'))\
                .agg(sum('prod_price').alias('year_prod_revenue'))\
            .withColumn('year_total_revenue',sum('year_prod_revenue').over(Window.partitionBy(col('year'))))\
            .withColumn('revenue_percentage',(col('year_prod_revenue')/col('year_total_revenue'))*100)\
            .withColumn('Dense_rank',dense_rank().over(Window.partitionBy('year').orderBy(col('revenue_percentage').desc())))\
            .orderBy(col('year').asc(),col('Dense_rank').asc(),col('product_id').asc())\
            .filter(col('Dense_rank')==1)\
            .show()

[Stage 253:=================================================>   (186 + 4) / 200]

+----------+--------------------+----+------------------+--------------------+------------------+----------+
|product_id|        product_name|year| year_prod_revenue|  year_total_revenue|revenue_percentage|Dense_rank|
+----------+--------------------+----+------------------+--------------------+------------------+----------+
|        93|   Smart Audio Mixer|2000|138247.14015197754|1.2038766014564514E7|1.1483497559860036|         1|
|        12| Power Bank 10000mAh|2001|145588.62001800537|1.1992513694969177E7|1.2139958620941944|         1|
|         8|           Webcam HD|2002| 137804.9301071167|1.2016051599266052E7| 1.146840365728239|         1|
|        14|USB Flash Drive 64GB|2003|139459.99993896484|1.1993534145317078E7| 1.162793203814887|         1|
|        34| WiFi Range Extender|2004|138160.97965240479|1.2163106137672424E7| 1.135902113231446|         1|
|        64|Smart Video Doorbell|2005| 137681.7296142578|1.2083784159446716E7|1.1393924932581871|         1|
|        85|Portabl

26/03/03 12:40:58 ERROR TaskSchedulerImpl: Lost executor 12 on 172.18.0.6: worker lost: Not receiving heartbeat for 60 seconds
26/03/03 12:40:58 ERROR TaskSchedulerImpl: Lost executor 13 on 172.18.0.4: worker lost: Not receiving heartbeat for 60 seconds


In [89]:
#testing for 2000 year

df_orders.filter(year(col('order_date'))==2000)\
            .groupBy(year(col('order_date')))\
            .agg(sum(col('price')*col('quantity')))\
            .show()

+----------------+-----------------------+
|year(order_date)|sum((price * quantity))|
+----------------+-----------------------+
|            2000|   1.2038766014564514E7|
+----------------+-----------------------+



In [ ]:
4. Using the orders and products datasets, calculate for each state the running cumulative revenue over time,
ordered by order_date, 
and return state, order_date, order_id, order_value, and cumulative revenue.

In [114]:
#orderid are not in ordered,so writing it in last
window_spec =Window.partitionBy('state').orderBy(col('order_date').asc(),col('order_id').asc())
df_orders.withColumn('order_value',col('price')*col('quantity'))\
            .withColumn('running_price',sum('order_value').over(window_spec).alias('running_price'))\
            .orderBy(col('state').asc(),col('order_date').asc(),col('order_id').asc())\
            .show()

[Stage 285:===================>                                  (72 + 4) / 200]

+--------+-----------+----------+------+----------+------------+--------------+--------+-----------+------------------+
|order_id|customer_id|product_id| price|order_date|order_status|         state|quantity|order_value|     running_price|
+--------+-----------+----------+------+----------+------------+--------------+--------+-----------+------------------+
|   71789|     710514|        24|157.53|2000-01-01|     SHIPPED|Andhra Pradesh|       2|     315.06|315.05999755859375|
|   86633|     241394|        43|130.78|2000-01-01|     SHIPPED|Andhra Pradesh|       2|     261.56| 576.6199951171875|
|  174827|     252523|        98|182.78|2000-01-01|      PLACED|Andhra Pradesh|       2|     365.56| 942.1799926757812|
|  228370|     384657|        84|169.01|2000-01-01|     SHIPPED|Andhra Pradesh|       1|     169.01|1111.1899871826172|
|  228859|     814445|        27| 143.7|2000-01-01|   DELIVERED|Andhra Pradesh|       1|      143.7|1254.8899841308594|
|  323726|     125335|        54|194.73|

26/03/03 14:12:16 ERROR TaskSchedulerImpl: Lost executor 14 on 172.18.0.6: worker lost: Not receiving heartbeat for 60 seconds
26/03/03 14:12:16 ERROR TaskSchedulerImpl: Lost executor 15 on 172.18.0.4: worker lost: Not receiving heartbeat for 60 seconds
26/03/03 14:43:18 ERROR TaskSchedulerImpl: Lost executor 16 on 172.18.0.4: worker lost: Not receiving heartbeat for 60 seconds
26/03/03 14:43:18 ERROR TaskSchedulerImpl: Lost executor 17 on 172.18.0.6: worker lost: Not receiving heartbeat for 60 seconds
26/03/03 15:14:20 ERROR TaskSchedulerImpl: Lost executor 18 on 172.18.0.6: worker lost: Not receiving heartbeat for 60 seconds
26/03/03 15:14:20 ERROR TaskSchedulerImpl: Lost executor 19 on 172.18.0.4: worker lost: Not receiving heartbeat for 60 seconds
26/03/03 15:45:22 ERROR TaskSchedulerImpl: Lost executor 20 on 172.18.0.6: worker lost: Not receiving heartbeat for 60 seconds
26/03/03 15:45:22 ERROR TaskSchedulerImpl: Lost executor 21 on 172.18.0.4: worker lost: Not receiving heartbeat

In [ ]:






5.For each product, find the first and last order date along with the total revenue generated between those dates, considering only non cancelled orders.

6. Identify customers whose latest order value is greater than their previous order value, and return customer_id, order_id, order_date, previous_order_value, and latest_order_value.

7.Using orders and products data, rank products within each year based on total profit, and return only the top 3 products per year.

8.For each state and order_status combination, calculate the percentage contribution of that combination to the total revenue of that state, and return state, order_status, revenue, and revenue_percentage.

9.For each customer, calculate the moving average of order value over their last 3 orders, ordered by order_date, and return customer_id, order_id, order_date, order_value, and moving_average.

10.Using orders and products datasets, find products whose average selling price is consistently higher than cost price across all years, and return product_id, average_unit_price, average_cost_price, and price_difference.